<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/08_practical_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08 · Practical project — a guided amplicon analysis

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: Python* - Instructor: Anderson Santos

## The scenario
You have just received the output of a 16S amplicon study: a **feature table** (`asv_table.csv`), a **taxonomy** (`taxonomy.csv`) and **sample metadata** (`sample_metadata.csv`). Your job is to take it from raw counts to a few defensible biological conclusions, the same path you will repeat all week.

## How this notebook works
It is a hands-on activity, not a lecture. Each step states a task and gives hints; you write the code in the empty cell. A collapsible solution sits under each step. Try first, then peek if stuck. The steps build on each other, so run them in order.

Skills you will combine: everything from notebooks 01–06 (variables, strings, loops, functions, and pandas).

> Suggested time: 45–60 min. Steps 1–6 are core; 7–9 are extension.

## Setup — run this cell first

This cell makes the example data available whether you are on **your own computer** or on **Google Colab**. You do not need to understand it. Click it and press **Shift+Enter**.

> **Instructor note.** Set `GITHUB_REPO` below to your repository once, before sharing the notebooks.

In [1]:
# Run me first. Works on a local install AND on Google Colab.
import os

GITHUB_REPO = "andersonavilasantos/ufz-summerschool-python"   # <-- set to your GitHub repo

if not os.path.exists("../data/asv_table.csv"):
    # Data not found locally -> assume Google Colab and download the course files.
    os.system(f"git clone -q https://github.com/{GITHUB_REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")

assert os.path.exists("../data/asv_table.csv"), (
    "Could not find the data. On Colab, check GITHUB_REPO above; "
    "locally, run the notebook from inside the notebooks folder.")
print("✅ Setup complete — the data folder is available.")

✅ Setup complete — the data folder is available.


> **Instructor note.** Run this as pair-programming: one drives, one navigates. Circulate and reveal each solution only after most pairs have attempted the step.

---
## Step 0 — Setup
Import the libraries and load the three tables. Nothing to solve here — just run it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

asv = pd.read_csv('../data/asv_table.csv', index_col='ASV_id')
taxonomy = pd.read_csv('../data/taxonomy.csv', index_col='ASV_id')
meta = pd.read_csv('../data/sample_metadata.csv', index_col='sample_id')
print('ASV table :', asv.shape)
print('taxonomy  :', taxonomy.shape)
print('metadata  :', meta.shape)

## Step 1 — Clean the feature table
**Task.** The raw table has missing values. (a) Report how many `NaN` cells there are, then (b) replace them with `0` and convert the table to integers. Store the result back in `asv`.

*Hints:* `.isnull().sum().sum()`, `.fillna(0)`, `.astype(int)`.

In [ ]:
# TODO Step 1

<details>
<summary><b>Solution: Step 1</b> (click to expand)</summary>

```python
print('NaN cells before:', int(asv.isnull().sum().sum()))
asv = asv.fillna(0).astype(int)
print('NaN cells after :', int(asv.isnull().sum().sum()))
```
</details>

## Step 2 — Quality control: library sizes
**Task.** Compute the **library size** (total reads) of each sample. (a) Print the smallest and largest, and (b) list any sample with fewer than **300** reads — these are candidates to drop.

*Hints:* `asv.sum(axis=0)` gives per-sample totals; boolean subsetting finds the small ones.

In [ ]:
# TODO Step 2

<details>
<summary><b>Solution: Step 2</b> (click to expand)</summary>

```python
lib = asv.sum(axis=0)
print('min library size:', lib.min(), ' max:', lib.max())
low = lib[lib < 300]
print('low-depth samples:', list(low.index))
```
</details>

> **Instructor note.** Real QC also plots a rarefaction curve; here we keep it to library sizes so it stays within the refresher toolkit.

## Step 3 — Filter rare ASVs (prevalence filter)
**Task.** Very rare features add noise. Keep only ASVs that are **present (count > 0) in at least 3 samples**. Report how many ASVs remain, and store the filtered table as `asv_f`.

*Hints:* `(asv > 0).sum(axis=1)` gives each ASV's prevalence; filter rows with a boolean mask.

In [ ]:
# TODO Step 3

<details>
<summary><b>Solution: Step 3</b> (click to expand)</summary>

```python
prevalence = (asv > 0).sum(axis=1)
asv_f = asv[prevalence >= 3]
print(f'kept {asv_f.shape[0]} of {asv.shape[0]} ASVs')
```
</details>

## Step 4 — Relative abundance
**Task.** Convert the filtered counts to **relative abundance** so samples are comparable regardless of sequencing depth. Verify each sample sums to 1. Store it as `rel`.

*Hints:* divide by the per-sample total (`axis=0`); check with `.sum(axis=0)`.

In [ ]:
# TODO Step 4

<details>
<summary><b>Solution: Step 4</b> (click to expand)</summary>

```python
rel = asv_f / asv_f.sum(axis=0)
print('all samples sum to 1?', np.allclose(rel.sum(axis=0), 1))
```
</details>

## Step 5 — Alpha diversity by environment
**Task.** For every sample compute (a) **observed richness** and (b) the **Shannon index**. Put them in a DataFrame `diversity`, join the metadata, and report the **mean of each metric per environment**. Which habitat is most diverse?

*Hints:* richness = `(asv_f > 0).sum(axis=0)`; write a `shannon(col)` function and `asv_f.apply(shannon, axis=0)`; then `groupby('environment')`.

In [ ]:
# TODO Step 5

<details>
<summary><b>Solution: Step 5</b> (click to expand)</summary>

```python
def shannon(col):
    p = col[col > 0] / col.sum()
    return -(p * np.log(p)).sum()

diversity = pd.DataFrame({
    'richness': (asv_f > 0).sum(axis=0),
    'shannon':  asv_f.apply(shannon, axis=0),
}).join(meta[['environment', 'treatment']])
print(diversity.groupby('environment')[['richness', 'shannon']].mean().round(2))
```
</details>

## Step 6 — Taxonomic composition per environment
**Task.** Collapse the relative-abundance table to **phylum** level and report the **mean composition per environment** (as %). Name the dominant phylum in each habitat.

*Hints:* join `taxonomy['Phylum']` onto `rel`, `groupby('Phylum').sum()`; then average per environment (transpose + join `meta['environment']`).

In [ ]:
# TODO Step 6

<details>
<summary><b>Solution: Step 6</b> (click to expand)</summary>

```python
rel_phylum = rel.join(taxonomy['Phylum']).groupby('Phylum').sum()
phylum_env = rel_phylum.T.join(meta['environment']).groupby('environment').mean().T
print((phylum_env * 100).round(1))
print('dominant phylum per environment:')
print(phylum_env.idxmax())
```
</details>

> **Instructor note.** Checkpoint: this table should show Cyanobacteriota high in Freshwater, Euryarchaeota high in Sediment, Acidobacteriota high in Soil.

## Step 7 — Treatment effect (extension)
**Task.** Does the **Amended** treatment enrich any phylum? Compute the mean relative abundance per **treatment** at phylum level, then the **Amended / Control fold-change**, and report the most enriched phylum.

*Hints:* group `rel_phylum` by treatment; divide the two rows; `idxmax()`.

In [ ]:
# TODO Step 7

<details>
<summary><b>Solution: Step 7</b> (click to expand)</summary>

```python
phylum_treat = rel_phylum.T.join(meta['treatment']).groupby('treatment').mean()
fold = phylum_treat.loc['Amended'] / phylum_treat.loc['Control']
print(fold.sort_values(ascending=False).round(2))
print('most enriched by amendment:', fold.idxmax())
```
</details>

## Step 8 — Correlation with an environmental gradient (extension)
**Task.** Pick the phylum you found most habitat-specific (e.g. *Cyanobacteriota*). Compute its per-sample relative abundance and its **Pearson correlation** with `pH`, `temperature_C` and `depth_cm`. Which variable does it track most strongly?

*Hints:* sum the phylum's rows in `rel`; `.corr()` against each metadata column.

In [ ]:
# TODO Step 8

<details>
<summary><b>Solution: Step 8</b> (click to expand)</summary>

```python
phy = 'Cyanobacteriota'
ids = taxonomy.index[taxonomy['Phylum'] == phy]
series = rel.loc[rel.index.intersection(ids)].sum(axis=0)
for var in ['pH', 'temperature_C', 'depth_cm']:
    print(f'{phy} vs {var}: r = {series.corr(meta[var]):.2f}')
```
</details>

## Step 9 — A summary figure (extension)
**Task.** Produce one clear figure that tells the story — for example a **stacked bar chart** of phylum composition per environment (from Step 6).

*Hints:* `phylum_env.T.plot(kind='bar', stacked=True)`; label axes and title.

In [ ]:
# TODO Step 9

<details>
<summary><b>Solution: Step 9</b> (click to expand)</summary>

```python
ax = (phylum_env * 100).T.plot(kind='bar', stacked=True, figsize=(9, 5))
ax.set_ylabel('Mean relative abundance (%)'); ax.set_xlabel('Environment')
ax.set_title('Phylum composition across environments')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()
```
</details>

## Step 10 — Export & conclude
**Task.** (a) Save your `diversity` table and the phylum-by-environment table to CSV. (b) In the markdown cell below, write **2–3 sentences** answering: *which environments differ most, and which taxa drive the difference?*

*Hints:* `.to_csv('...')`.

In [ ]:
# TODO Step 10 (export)

<details>
<summary><b>Solution: Step 10</b> (click to expand)</summary>

```python
diversity.to_csv('project_diversity.csv')
(phylum_env * 100).round(2).to_csv('project_phylum_by_environment.csv')
print('exported project_diversity.csv and project_phylum_by_environment.csv')
```
</details>

### Your conclusions
*(double-click to edit this cell and write your interpretation)*

- Environments that differ most: ...
- Taxa driving the difference: ...
- One thing I would check next: ...

---
## Wrap-up
You just ran a complete, if compact, microbial-ecology workflow: clean, QC, filter, normalise, diversity, composition, statistics, figure, export. Every tool here came from the refresher notebooks. The only new thing was chaining them into an analysis.

Keep this notebook as a template: swap in a real feature table this week and most of the code will still apply.